# 组件模板（TransmonPocket）DSL 演示

用 `type: transmon_pocket` 模板自动生成两个 transmon qubit 的几何和 pin。

对应 YAML：`transmon_pocket_2q.metal.yaml`

和 primitive-native（见 `primitive_native_demo.ipynb`）不同，
这里只写 type 和 options——模板负责生成 pad、pocket、junction 和 connection pad 的全部几何。

In [ ]:
from pathlib import Path
import sys

_SRC = Path("../../src").resolve()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from qiskit_metal.toolbox_metal.design_dsl import build_ir, build_design

yk07-main\src\qiskit_metal\_gui\styles\metal_dark\rc\transparent.png (No context available from Qt)Odeircuit\qiskit\qiskit-metal-worktrees
Python Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\envs\metal-env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\envs\metal-env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\envs\metal-env\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\envs\metal-env\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "c:\ProgramData\anaconda3\envs\metal-env\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "c:\ProgramData\anaconda3\envs\metal-env\Lib\asyncio\base_events.py", line 1936, in _run_once
    handle._run()
  F

## 1. 构建

`build_ir()` 遇到 `type: transmon_pocket` 时，沿着继承链展开模板：

```
qcomponent.yaml → base_qubit.yaml → transmon_pocket.yaml
```

每一层提供不同层次的默认值和生成规则。

In [2]:
yaml_file = Path("transmon_pocket_2q.metal.yaml")

ir = build_ir(yaml_file)
design = build_design(yaml_file)
chain = design.metadata["dsl_chain"]

names = [c.name for c in ir.components]
print(f"schema:     {ir.schema}")
print(f"components: {names}")

schema:     qiskit-metal/design-dsl/3
components: ['Q1', 'Q2']


In [3]:
from qiskit_metal import MetalGUI
gui = MetalGUI(design)

## 2. 模板继承

每个 component 的 metadata 记录了它经过的模板链。
注意导出的 class 是 `NativeComponent`，不是 qlibrary 的 `TransmonPocket`。

In [4]:
for name in names:
    meta = design.components[name].metadata["template"]
    cls = design.components[name].__class__.__name__
    print(f"{name}: {meta['inherited']}  →  {cls}")

Q1: ['qcomponent', 'base_qubit', 'transmon_pocket']  →  NativeComponent
Q2: ['qcomponent', 'base_qubit', 'transmon_pocket']  →  NativeComponent


## 3. 生成的几何

模板自动生成了：
- **poly**: pad_top、pad_bot（两块 pad）、rect_pk（pocket 挖空）、readout_connector_pad
- **path**: readout_wire、readout_wire_sub
- **junction**: rect_jj

In [5]:
for table in ("poly", "path", "junction"):
    rows = sorted(design.qgeometry.tables[table]["name"].tolist())
    print(f"{table:10s}: {len(rows)} 行  {rows}")

poly      : 8 行  ['pad_bot', 'pad_bot', 'pad_top', 'pad_top', 'readout_connector_pad', 'readout_connector_pad', 'rect_pk', 'rect_pk']
path      : 4 行  ['readout_wire', 'readout_wire', 'readout_wire_sub', 'readout_wire_sub']
junction  : 2 行  ['rect_jj', 'rect_jj']


展开看看 poly 表——注意每个 component 都有自己的一组行：

In [6]:
design.qgeometry.tables["poly"][["component", "name", "subtract", "layer"]]

,component,name,subtract,layer
0,1,pad_top,False,1
1,1,pad_bot,False,1
2,1,rect_pk,True,1
3,1,readout_connector_pad,False,1
4,2,pad_top,False,1
5,2,pad_bot,False,1
6,2,rect_pk,True,1
7,2,readout_connector_pad,False,1


## 4. 生成的 pin 和连接

YAML 里写了 `connection_pads: readout`，模板就生成了 `readout` pin。
两个 qubit 的 readout 连在同一个 net 上。

In [7]:
for name in names:
    pins = sorted(design.components[name].pins)
    print(f"{name} pins: {pins}")

print(f"\nnet_info ({len(design.net_info)} 行):")

Q1 pins: ['readout']
Q2 pins: ['readout']

net_info (2 行):


In [8]:
design.net_info

,net_id,component_id,pin_name
0,1,1,readout
1,1,2,readout


In [9]:
q1_net = int(design.components["Q1"].pins["readout"].net_id)
q2_net = int(design.components["Q2"].pins["readout"].net_id)
print(f"Q1.readout net_id = {q1_net}")
print(f"Q2.readout net_id = {q2_net}")
assert q1_net == q2_net
print("共享同一个 net")

Q1.readout net_id = 1
Q2.readout net_id = 1
共享同一个 net


## 5. 检查

In [10]:
types = {n: chain["geometry"]["components"][n].get("type") for n in names}
assert names == ["Q1", "Q2"]
assert types == {"Q1": "transmon_pocket", "Q2": "transmon_pocket"}
assert all(
    design.components[n].__class__.__name__ == "NativeComponent"
    for n in names
)

poly_names = sorted(design.qgeometry.tables["poly"]["name"].tolist())
assert poly_names.count("pad_top") == 2
assert poly_names.count("readout_connector_pad") == 2
assert len(design.net_info) == 2

print("全部通过")

全部通过
